## В этом домашнем задании вы сделаете первые шаги в мире линейной бинарной классификации!

In [ ]:
import pandas as pd
import numpy as np
import warnings

pd.options.display.max_columns = 500

#### Задание 1

Мы будем работать с данными **Microsoft Malware Detection**

Таргетом будет последний столбец `HasDetection`, который принимает значения $\{0,\, 1\}$ в случае отсутствия или наличия вируса на компьютере соответственно. Признаками будут выступать всевозможные характеристики коспьютера.

In [ ]:
### Загрузим датасет

data = pd.read_csv('train.csv')

In [ ]:
data.head()

Удалите константные признаки и признаки `ProductName` `MachineIdentifier`

In [ ]:
### Можно воспользоваться VarianceThreshold
### А можно еще проще с помощью скрина метода .all()

for i in data.columns:
    
    if (data[i] == data[i][0]).all():
        data.drop(columns=[i], inplace=True)

In [ ]:
### Дополнительно удалим еще 2 колонки

data.drop(columns=['ProductName', 'MachineIdentifier'], inplace=True)

Посмотрите на соотношение классов в таргете. Все ли хорошо?

In [ ]:
### Посмотрим, сколько наблюдений в датасете 
### имеют 0 и 1 классы

print(sum(data['HasDetections'] == 1), '- positive class,')
print(sum(data['HasDetections'] == 0), '- negative class')

### Кажется, с балансом классов все ок!

Ответьте на вопрос: почему с вашей точки зрения важно иметь представление о балансе классов в ваших данных?

**При дисбалансе модели могут подстраиваться под доминирующий класс, когда для нас может быть интересен как раз-таки другой**

Избавьтесь от пропусков в данных! 

Новый для нас прием: признаки с более чем половиной пропусков следует удалить.

Согласитесь, если в вашей колоночке среди 100 объектов всего лишь у 2 есть какое-то непропущенное значение, странно все остальные заполнять средним от этих двух чисел. Такие "редкие" признаки лучше вообще опустить!


В категориальных колонках заменим отсутствующую категорию просто некоторой новой и назовем ее `NaN`.

А в числовых, ради разнообразия, заполним пропуски медианным значением.

In [ ]:
### Посчитаем количество отсутствующих значений в каждой колонке

nans = np.sum(data.isna(), axis=0)

### Удалим те колонки, в которых пропусков более половины от размера всей выборки

data.drop(columns=list(nans[nans > data.shape[0] / 2].index), inplace=True)

In [ ]:
### В категориальных колонках с отсутствующими значениями
### произведем fillna, создав по новой категории для каждой
### такой фичи

cat_cols = data.select_dtypes(exclude=['float64', 'int64']).isnull().any()

for null_col in cat_cols[cat_cols].index:
    data[null_col] = data[null_col].fillna('NaN')


In [ ]:
### Оставшиеся (вещественные) колонки заполним медианой по столбцу

nans = np.sum(data.isna(), axis=0)


for col in list(nans[nans > 0].index):
    data[col].fillna(data[col].median(), inplace=True)

In [ ]:
data.head()

Создайте копию полученного датафрейма и положите ее в переменную data_2. Понадобится в следующих заданиях.

In [ ]:
data_2 = data.copy()

Так же поработаем над всеми категориальными колонкам перед запуском непосредственно моделей.

Провернем самый базовый и наглый метод - несмотря на количество уникальных значений в каждой категории, просто применим ко всей категориальной части датасета `OneHotEncoding`

In [ ]:
### Для OneHotEncoding'а воспользуемся пандасовским get_dummies

cat_cols = data.select_dtypes(exclude=['float64', 'int64']).columns

categorical_data_part = pd.get_dummies(data[cat_cols],
                                       prefix=cat_cols,
                                       drop_first=True,
                                       )

data = data.drop(cat_cols, axis=1)

data = pd.concat((data, categorical_data_part), axis=1)

###  Разделим выборку на тренировочную и тестовую

P.S. в задачах классификации, как и в задаче регрессии, можно использовать технологию Кросс-Валидации. Например, по одному из двух следующих сценариев:

1) Отделить валидацию и тест, произвести подбор лучшей модели с помощью `K-Fold` на валидации, финально обучить выбранную модель на всей валидации и замерить качество на заранее отложенном финальном тесте!

2) Всю выборку назвать валидационной и на ней применить `K-Fold` без финального замера.

В этой домашней работе попросим Вас быть еще проще! :)
Реализуем просто технологию отложенной выборки в пропорции 3:1

In [ ]:
### Разобьем выборку на трейн и тест

from sklearn.model_selection import train_test_split 

X = data.drop(columns=['HasDetections'])
y = data['HasDetections']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.25,
                                                    shuffle=True,
                                                    random_state=1)

Соберите `Pipeline`, реализовав в нем 2 шага: стандартизация данных через `StandardScaler` и обучение логистической регрессии с помощью `LogisticRegression`, положите результаты в переменную `pipe`, а в классе модели `LogisticRegression` укажите параметр `penalty='none'`

In [ ]:
### Здесь все просто и знакомо

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipe = Pipeline([('scaler', StandardScaler()), 
                 ('LR', LogisticRegression(penalty='none'))])

Чтобы замерить качество работы такой модели на трейне и на тесте воспользуемся функцией `cross_validate`

Вопрос: что передавать ей в параметр cv? Ведь мы уже разделили нашу выборку на трейн и тест (имеем всего 1 fold). Для этого можно просто передать список от кортежа, содержащего индексы тренировочных и тестовых объектов.

In [ ]:
from sklearn.model_selection import cross_validate
import datetime

### Создадим такой список из одного единственного кортежа
### Первый элемент - индексы обучающей выборки
### Второй элемент - индексы тестовой выборки

custom_cv = [(X_train.index.to_list(), X_test.index.to_list())]

### Замерим начало работы cross_validate

begin_time = datetime.datetime.now()

### В cv передадим созданный custom_cv
### В качестве выборки передадим все данные
### Метод сам разобьет выборку на трейн и тест по нашей стратегии

cv_result_pipe = cross_validate(pipe, X, y, scoring='accuracy',
                                cv=custom_cv, return_train_score=True)


print(f"Accuracy на трейне: {np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Accuracy на тесте: {np.mean(cv_result_pipe['test_score']).round(3)}")

print(f"Время работы алгоритма: {datetime.datetime.now() - begin_time}")

Что можете сказать про время работы алгоритма?

Очевидно, оно достаточно большое. Уж тем более для линейных моделей.

Такое как раз-таки происходит из-за того, что количество фичей, который мы передали нашей модели - гигантское! Классу требуется много времени и памяти, чтобы обработать датасет.

Поэтому те колонки, в которых количество уникальных категорий превышает какое-то адекватное число, следует кодировать иначе, нежели с помощью технологии `One-Hot-Encoding`.

Теперь вы верите, что более умные кодировки зачастую прям необходимы! Раньше мы этот факт не демонстрировали!

Дело еще вот в чем: в классе `LogisticRegression`, как и, например, `Lasso`, есть параметр, ограничивающий максимальное количество итераций во время обучения модели. Так, если данных много и итераций тоже ожидается большое число, найденная разделяющая гиперплоскость может оказаться не самой лучшей, так как наш алгоритм (будь то градиентный спуск или любой иной) просто 'не доползет'. 

#### Задание 2

Теперь попробуем другой метод кодирования категориальных колонок, а именно счётчики.
Построем ту же модель и на том же разделении, просто заново иначе переобработаем датасет. 

Для тех категориальных признаков, у которых количество уникальных значений в колоночках больше 5, применим `MeanTargetEncoding`.

Для остальных оставим любимый `OneHotEncoding` (как делали на практике и в предыдущем уроке).

In [ ]:
### Закодируем категории с большим кол-ом уникальных значений с помощью счетчиков
### Закрывая глаза на возможную проблему переобучения
### То есть будем считать средние по всему датасету

for col in cat_cols:
    if data_2[col].nunique() < 5:
        one_hot = pd.get_dummies(data_2[col], prefix=col, drop_first=True)
        data_2 = pd.concat((data_2.drop(col, axis=1), one_hot), axis=1)
        
    else:
        mean_target = data_2.groupby(col)['HasDetections'].mean()
        data_2[col] = data_2[col].map(mean_target)

In [ ]:
X_2 = data_2.drop(columns=['HasDetections'])
y_2 = data_2['HasDetections']

Опять обучим модель, пока что без изменений! Скажите, стало ли быстрее? А что с качеством?

In [ ]:
### Заново обучим модель и провалидируем
### Очевидно стало супер-быстрее!

begin_time = datetime.datetime.now()

cv_result_pipe = cross_validate(pipe, X_2, y_2, scoring='accuracy',
                                cv=custom_cv, return_train_score=True)

print(f"Accuracy на трейне: {np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Accuracy на тесте: {np.mean(cv_result_pipe['test_score']).round(3)}")

print(f"Время работы алгоритма: {datetime.datetime.now() - begin_time}")

#### Задание 3: Регуляризация

Как и в моделях регрессии, решая задачу классификации, можем штрафовать минимизируемый функционал за большие веса, добавив к нему L1 или L2 норму весов (все как раньше!).

Для этого в изначальном классе `LogisticRegression` изменить параметр `penalty` на l1 или l2 соответственно. Выберите второй вариант! Можно воспользоваться методом `set_params` и применить его к `pipe`.

In [ ]:
### Установим в качестве параметра penalty
### L2 регуляризатор!

pipe.set_params(LR__penalty='l2')

Теперь наша модель будет строить логистическую регрессию с L1 регуляризатором! Помним, что у регуляризируемых моделей есть гиперпараметр, контролирующий силу регуляризации, который выбирается ДО запуска метода fit, то есть заранее, когда модель еще не обучена. 

Конечно же, выбор этого параметра влияет итоговые результаты. Хочется поперебирать различные параметры регуляризации и найти такой, при котором качество на тесте окажется лучшим! 

Сгенерируем массив гиперпараметра, которые планируем перебрать:

In [ ]:
### Сгенерим кучу параметров для регуляризации

alphas = np.linspace(0.01, 100, 100)
alphas

Чтобы отобрать среди данного массива гиперпараметров лучший, воспользуйтесь конструкцией `GridSearchCV` из `sklearn`

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid ={
    'LR__C': alphas
}


search_alpha = GridSearchCV(pipe, param_grid, 
                            cv=custom_cv, scoring='accuracy')

search_alpha.fit(X_2, y_2)

print(f"Best parameter (CV score={search_alpha.best_score_:.5f}):")
print(search_alpha.best_params_)

#### Задание 4: Бонус

Как вы знаете, подбор признаков является одной из самых важных частей машинного обучения. Сейчас вы попробуете на основе имеющихся признаков сгенерировать новые. В качестве новых признаков будем использовать мономы 2 степени. Можно использовать регуляризацию различного рода, выбор энкодера тоже за вами. Ваша задача - добиться качества `0.65` на тестовой выборке

### С помощью мономов 2 степени легко добиться качества > 0.643

In [ ]:
### Your code is here

X_3, y_3 = X_2.copy(), y_2.copy()

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(2)

X_3 = poly.fit_transform(X_3)

In [ ]:
alphas = [0.001, 0.005, 0.01]

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid ={
    'LR__C': alphas
}


search_alpha = GridSearchCV(pipe, param_grid, 
                            cv=custom_cv, scoring='accuracy')

search_alpha.fit(X_3, y_3)

print(f"Best parameter (CV score={search_alpha.best_score_:.5f}):")
print(search_alpha.best_params_)

### Чтобы все же пробить бейзлайн из бонусного задания, придумаем новые фичи!

In [ ]:
### Заново загрузим датасет

df = pd.read_csv('train.csv')

df = df.drop(['ProductName', 'MachineIdentifier'], axis=1)

df.head()

In [ ]:
X_new = df.drop(columns=['HasDetections'])
y_new = df['HasDetections']

### Создадим 2 новые фичи
### Подумайте, что они значат и почему могут быть полезны в задаче?

X_new['driveA'] = X_new['Census_SystemVolumeTotalCapacity'].astype('float')/X_new['Census_PrimaryDiskTotalCapacity'].astype('float')
X_new['driveA'] = X_new['driveA'].astype('float32') 


X_new['driveB'] = X_new['Census_PrimaryDiskTotalCapacity'].astype('float') - X_new['Census_SystemVolumeTotalCapacity'].astype('float')
X_new['driveB'] = X_new['driveB'].astype('float32') 

In [ ]:
### Разделим выборку на трейн и тест как раньше

X_train, X_test, y_train, y_test = train_test_split(X_new, y_new,
                                                    test_size=0.25,
                                                    shuffle=True,
                                                    random_state=1)

custom_cv = [(X_train.index.to_list(), X_test.index.to_list())]

In [ ]:
### Сделаем препроцессинг 1 в 1 как раньше
### Аналогично поработав с пропусками и с категориями

### Посчитаем количество отсутствующих значений в каждой колонке

nans = np.sum(X_new.isna(), axis=0)

### Удалим те колонки, в которых пропусков более половины от размера всей выборки

X_new.drop(columns=list(nans[nans > data.shape[0] / 2].index), inplace=True)

### В категориальных колонках с отсутствующими значениями
### произведем fillna, создав по новой категории для каждой
### такой фичи

cat_cols = X_new.select_dtypes(exclude=['float64', 'int64']).isnull().any()

for null_col in cat_cols[cat_cols].index:
    X_new[null_col] = X_new[null_col].fillna('NaN')

### Оставшиеся (вещественные) колонки заполним медианой по столбцу

nans = np.sum(X_new.isna(), axis=0)


for col in list(nans[nans > 0].index):
    X_new[col].fillna(X_new[col].median(), inplace=True)
    
data_new = pd.concat((X_new, pd.concat((y_train, y_test))), axis=1)

### Закодируем категории с большим кол-ом уникальных значений с помощью счетчиков
### Закрывая глаза на возможную проблему переобучения
### То есть будем считать средние по всему датасету

cat_cols = X_new.select_dtypes(exclude=['float64', 'int64']).columns

for col in cat_cols:
    if data_new[col].nunique() < 5:
        one_hot = pd.get_dummies(data_new[col], prefix=col, drop_first=True)
        data_new = pd.concat((data_new.drop(col, axis=1), one_hot), axis=1)
        
    else:
        mean_target = data_new.groupby(col)['HasDetections'].mean()
        data_new[col] = data_new[col].map(mean_target)

In [ ]:
from sklearn.model_selection import GridSearchCV

X_new = data_new.drop(columns=['HasDetections'])
y_new = data_new['HasDetections']

param_grid ={
    'LR__C': alphas
}

search_alpha = GridSearchCV(pipe, param_grid, 
                            cv=custom_cv, scoring='accuracy')

search_alpha.fit(X_new, y_new)

print(f"Best parameter (CV score={search_alpha.best_score_:.5f}):")
print(search_alpha.best_params_)

Видно, что два новых признака существенно улучшили модель! Если добавить сюда мономы - качество окажется еще лучше!